# Анализ на искане за възстановяване на разходи

Този бележник демонстрира как да се създават агенти, които използват приставки за обработка на разходи за пътуване от локални изображения на касови бележки, генерират имейл за искане на възстановяване на разходи и визуализират данните за разходите чрез диаграма тип пай. Агентите динамично избират функции въз основа на контекста на задачата.

Стъпки:
1. OCR агент обработва локалното изображение на касовата бележка и извлича данни за разходите за пътуване.
2. Email агент генерира имейл за искане на възстановяване на разходи.

### Пример за сценарий с разходи за пътуване:
Представете си, че сте служител, който пътува за бизнес среща в друг град. Вашата компания има политика да възстановява всички разумни разходи, свързани с пътуването. Ето разбивка на потенциалните разходи за пътуване:
- Транспорт:
Самолетни билети за двупосочно пътуване от вашия град до града-дестинация.
Такси или услуги за споделено пътуване до и от летището.
Местен транспорт в града-дестинация (като обществен транспорт, коли под наем или таксита).

- Настаняване:
Настаняване в хотел за три нощувки в бизнес хотел от среден клас, близо до мястото на срещата.

- Храна:
Дневно обезщетение за храна за закуска, обяд и вечеря, базирано на дневната политика на компанията.

- Разни разходи:
Такси за паркиране на летището.
Такси за достъп до интернет в хотела.
Бакшиши или малки услуги.

- Документация:
Вие подавате всички касови бележки (за полети, таксита, хотел, храна и др.) и попълнен отчет за разходите за възстановяване.


## Импортирайте необходимите библиотеки

Импортирайте необходимите библиотеки и модули за тетрадката.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Дефиниране на модели за разходи

Създайте Pydantic модел за индивидуални разходи и клас ExpenseFormatter за преобразуване на заявка от потребителя в структурирани данни за разходи.

Всеки разход ще бъде представен във формата:
 `{'date': '07-Mar-2025', 'description': 'полет до дестинация', 'amount': 675.99, 'category': 'Транспорт'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Дефиниране на инструменти - Генериране на имейл

Създайте функция на инструмент за генериране на имейл за подаване на искане за възстановяване на разходи.
- Този инструмент използва декоратора `@tool` от Microsoft Agent Framework.
- Той изчислява общата сума на разходите и форматира детайлите в тяло на имейл.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Инструмент за извличане на разходи за пътувания от снимки на касови бележки

Създайте функция за инструмент за извличане на разходи за пътувания от снимки на касови бележки.
- Този инструмент използва `@tool` декоратора от Microsoft Agent Framework.
- Той прочита снимката на касовата бележка, кодира я като base64 и връща data URI за анализ от агента.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Обработка на разходите

Дефинирайте агентите и ги свържете в последователен работен процес, използвайки `WorkflowBuilder`.
- OCR агентът извлича структурирани данни за разходите от изображението на касовата бележка, използвайки инструмента `load_receipt_image`.
- Имейл агентът взима извлечените данни и генерира професионален имейл за искане на възстановяване на разходи, използвайки инструмента `generate_expense_email`.
- `WorkflowBuilder` с `add_edge` създава последователен поток: OCR агент → Имейл агент.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Основна функция

Създайте последователния работен процес и го изпълнете, за да обработите изображението на касовата бележка и да генерирате имейл за претенция за разходи.


> **Бележка:** Този работен процес в момента предава изображението на касовата бележка като base64 текст, което повечето чат модели (включително gpt-4o) няма да третират като изображение.  
> Също така може да надхвърли контекстното поле на модела. Предпочитайте да изпълните OCR с Azure AI Vision (или друг OCR инструмент) и да предавате само извлечения текст, или преструктурирайте така, че да изпращате изображението като съобщение `image_url`.  
> Ако искате само да избегнете грешки с контекста, опитайте с по-малко изображение на касовата бележка или модел с по-голямо контекстно поле.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
